# Day 085 Project — MCP-Connected Assistant

Build an MCPServer with three tools, connect an MCPAgent to it, and run several queries through the full MCP stack.

In [ ]:
import json
from dataclasses import dataclass, field
@dataclass
class MCPToolDef:
    name: str
    description: str
    input_schema: dict = field(default_factory=dict)

def tool_schema_text(tools):
    lines = []
    for t in tools:
        params = ", ".join(t.input_schema.keys())
        lines.append("- " + t.name + "(" + params + "): " + t.description)
    return "\n".join(lines)
class MCPServer:
    def __init__(self, name="mcp_server"):
        self.name = name
        self._tools = {}
    def tool(self, name, description, schema=None):
        def _decorator(fn):
            self._tools[name] = {"def": MCPToolDef(name, description, schema or {}), "fn": fn}
            return fn
        return _decorator
    def list_tools(self):
        return [e["def"] for e in self._tools.values()]
    def call_tool(self, name, args):
        e = self._tools.get(name)
        if e is None:
            return "Error: unknown tool " + repr(name)
        try:
            return str(e["fn"](**args))
        except Exception as exc:
            return "Error: " + str(exc)
def call_llm(messages, llm_fn=None):
    if llm_fn is not None:
        return str(llm_fn(messages))
    import ollama
    resp = ollama.chat(model="llama3.2", messages=messages)
    return resp["message"]["content"]

def safe_parse_json(text):
    start = str(text).find("{")
    end   = str(text).rfind("}") + 1
    if start == -1 or end == 0:
        return None
    try:
        return json.loads(text[start:end])
    except (json.JSONDecodeError, ValueError):
        return None
class MCPClient:
    def __init__(self, server=None, tool_call_fn=None):
        self._server = server
        self._tool_call_fn = tool_call_fn
    def list_tools(self):
        if self._server is not None:
            return self._server.list_tools()
        return []
    def call_tool(self, name, args):
        if self._tool_call_fn is not None:
            return self._tool_call_fn(name, args)
        if self._server is not None:
            return self._server.call_tool(name, args)
        return "Error: no server or tool_call_fn configured"
def build_mcp_selection_prompt(query, tools):
    menu = tool_schema_text(tools)
    system = "\n".join([
        "You are a tool router. Pick the best tool for the request.",
        "Available tools:", menu,
        "Return ONLY a JSON object with keys 'tool' and 'args'.",
        "Use tool name 'none' if no tool fits.",
    ])
    return [{"role": "system", "content": system},
            {"role": "user",   "content": "Request: " + str(query)}]

def select_mcp_tool(query, client, llm_fn=None):
    tools = client.list_tools()
    if not tools:
        return {"tool": "none", "args": {}}
    messages = build_mcp_selection_prompt(query, tools)
    response = call_llm(messages, llm_fn=llm_fn)
    data = safe_parse_json(response) or {}
    name = data.get("tool", "none")
    args = data.get("args", {})
    known = {t.name for t in tools}
    if name not in known:
        name = "none"
    return {"tool": name, "args": args if isinstance(args, dict) else {}}
def _mock_mcp_llm(tool="none", args=None):
    payload = json.dumps({"tool": tool, "args": args or {}})
    return lambda messages: payload

def _mock_tool_call(name, args):
    return "Result:" + str(name)

# ── Copy of MCPAgent (from mcp_agent.py) ────────────────────────────────────
class MCPAgent:
    def __init__(self, client, llm_fn=None):
        self.client = client
        self._llm_fn = llm_fn
        self._history = []
    def tools(self): return self.client.list_tools()
    def ask(self, query):
        selection = select_mcp_tool(query, self.client, llm_fn=self._llm_fn)
        if selection["tool"] == "none":
            record = {"query": query, "tool": "none", "args": {}, "result": "No suitable tool found."}
        else:
            result = self.client.call_tool(selection["tool"], selection["args"])
            record = {"query": query, "tool": selection["tool"], "args": selection["args"], "result": result}
        self._history.append(record)
        return record
    def history(self): return list(self._history)
    def clear_history(self): self._history.clear()


## Step 1 — Build the Server

Register three tools on an MCPServer.

In [ ]:
server = MCPServer("assistant_server")

@server.tool("word_count", "Count the number of words in text.", {"text": "the text to count"})
def word_count(text):
    return str(len(str(text).split()))

@server.tool("uppercase", "Convert text to uppercase.", {"text": "the text to convert"})
def uppercase(text):
    return str(text).upper()

@server.tool("reverse_words", "Reverse the word order in text.", {"text": "the text to reverse"})
def reverse_words(text):
    return " ".join(reversed(str(text).split()))

print("Registered tools:", [t.name for t in server.list_tools()])


## Step 2 — Create Client and Agent

Connect a client to the server, then wire it into an MCPAgent.

In [ ]:
client = MCPClient(server=server)

# For gate testing we inject a mock LLM.
# Remove llm_fn=... and add llm_fn=None (or omit it) to use real Ollama.
llm_fn = _mock_mcp_llm("word_count", {"text": "the quick brown fox"})

agent = MCPAgent(client, llm_fn=llm_fn)
print("Agent tools:", [t.name for t in agent.tools()])


## Step 3 — Run Queries

In [ ]:
queries = [
    "How many words are in 'the quick brown fox'?",
]

for q in queries:
    r = agent.ask(q)
    print(f"Q: {r['query']}")
    print(f"  Tool: {r['tool']}  Args: {r['args']}")
    print(f"  Result: {r['result']}")
    print()


## Step 4 — Inspect History

In [ ]:
print(f"Total interactions: {len(agent.history())}")
for i, entry in enumerate(agent.history(), 1):
    print(f"{i}. [{entry['tool']}] {entry['result'][:60]}")
